# Análisis exploratorio de datos — Anuncios de venta de coches en EE. UU.

Este cuaderno recoge la exploración previa del conjunto de datos `vehicles_us.csv`
que sirve de base para la aplicación web construida con Streamlit.

El objetivo no es un análisis exhaustivo, sino **entender la forma de los datos y
probar los gráficos de Plotly** que después se integran en el panel de control.

**Aplicación desplegada:** ver el enlace en el `README.md` del repositorio.

## 1. Carga de los datos

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Paleta validada para daltonismo, la misma que usa la aplicación
AZUL, NARANJA, MORADO = '#3b6fd4', '#e08b2f', '#8a5cd0'

# El notebook vive en notebooks/, el CSV está en la raíz del proyecto
car_data = pd.read_csv('../vehicles_us.csv')

print('Filas y columnas:', car_data.shape)

In [ ]:
car_data.head(10)

In [ ]:
car_data.info()

In [ ]:
car_data.describe()

## 2. Calidad de los datos

Reviso los valores ausentes y los posibles valores atípicos antes de graficar
nada, porque ambos condicionan cómo se ven los gráficos.

In [ ]:
# Valores ausentes por columna
ausentes = pd.DataFrame({
    'ausentes': car_data.isna().sum(),
    'porcentaje': (100 * car_data.isna().mean()).round(2)
})
print(ausentes[ausentes['ausentes'] > 0])

In [ ]:
# Valores extremos que conviene conocer
print('Precio      — mín: {}   máx: {}   mediana: {}'.format(
    car_data['price'].min(), car_data['price'].max(), car_data['price'].median()))
print('Odómetro    — mín: {}   máx: {}   mediana: {}'.format(
    car_data['odometer'].min(), car_data['odometer'].max(), car_data['odometer'].median()))
print('Año modelo  — mín: {}   máx: {}'.format(
    car_data['model_year'].min(), car_data['model_year'].max()))
print()
print('Anuncios con precio menor de 100 USD:', (car_data['price'] < 100).sum())
print('Anuncios con más de 500 000 millas:', (car_data['odometer'] > 500000).sum())

### Observaciones sobre la calidad de los datos

**Cinco columnas tienen valores ausentes:**

| Columna | Ausentes | Interpretación |
|---|---:|---|
| `is_4wd` | 25 953 (50,4 %) | El valor solo aparece como `1.0`: el hueco significa **"no es 4x4"**, no un dato perdido |
| `paint_color` | 9 267 (18,0 %) | El anunciante no declaró el color |
| `odometer` | 7 892 (15,3 %) | Kilometraje no informado |
| `cylinders` | 5 260 (10,2 %) | Sin especificar |
| `model_year` | 3 619 (7,0 %) | Año del modelo no informado |

**`is_4wd` es el caso más interesante**: no es información perdida sino una
codificación implícita. En la aplicación lo relleno con `0` y lo convierto a
entero, porque el hueco *sí* tiene significado.

**Hay valores atípicos evidentes:** precios desde 1 USD hasta 375 000, y
odómetros de hasta 990 000 millas. Son anuncios reales pero extremos, y aplastan
la escala de los gráficos. Por eso la aplicación incorpora un **filtro de precio**
que por defecto se limita a 60 000 USD.

También aparece un `model_year` de **1908**, probablemente un vehículo de colección
o un error de tecleo.

## 3. Histograma con Plotly

Primer gráfico del panel: la distribución del kilometraje.

In [ ]:
# Histograma del odómetro con plotly.graph_objects
fig = go.Figure(data=[go.Histogram(x=car_data['odometer'], marker_color=AZUL)])

fig.update_layout(
    title_text='Distribución del kilometraje',
    xaxis_title='Kilometraje (millas)',
    yaxis_title='Número de anuncios',
    bargap=0.02)

fig.show()

In [ ]:
# La misma idea con plotly.express, que es la sintaxis usada en la aplicación
fig = px.histogram(
    car_data,
    x='odometer',
    nbins=50,
    title='Distribución del kilometraje (plotly express)',
    labels={'odometer': 'Kilometraje (millas)'},
    color_discrete_sequence=[AZUL])

fig.update_layout(yaxis_title='Número de anuncios', bargap=0.02)
fig.show()

**Lectura del histograma.** La distribución está claramente **sesgada a la
derecha**: la mayoría de los vehículos se concentra entre las 50 000 y las
150 000 millas, con una mediana de unas 113 000, y a partir de ahí una cola larga
de coches muy rodados. También destaca un pico en el cero, correspondiente a
vehículos nuevos o a anuncios con el kilometraje mal informado.

## 4. Gráfico de dispersión con Plotly

Segundo gráfico del panel: la relación entre kilometraje y precio.

In [ ]:
# Dispersión odómetro frente a precio con plotly.graph_objects
fig = go.Figure(data=[go.Scatter(
    x=car_data['odometer'],
    y=car_data['price'],
    mode='markers',
    marker=dict(color=AZUL, opacity=0.35))])

fig.update_layout(
    title_text='Relación entre kilometraje y precio',
    xaxis_title='Kilometraje (millas)',
    yaxis_title='Precio (USD)')

fig.show()

In [ ]:
# Correlación entre ambas variables
correlacion = car_data['odometer'].corr(car_data['price'])
print('Correlación entre kilometraje y precio: {:+.4f}'.format(correlacion))

# Recortando los valores extremos se aprecia mucho mejor la relación
recorte = car_data[(car_data['price'] < 60000) & (car_data['odometer'] < 400000)]

fig = px.scatter(
    recorte,
    x='odometer',
    y='price',
    opacity=0.3,
    title='Kilometraje frente a precio (sin valores extremos)',
    labels={'odometer': 'Kilometraje (millas)', 'price': 'Precio (USD)'},
    color_discrete_sequence=[AZUL])

fig.show()
print('Correlación sin los extremos: {:+.4f}'.format(
    recorte['odometer'].corr(recorte['price'])))

**Lectura del gráfico de dispersión.** La correlación entre kilometraje y
precio es **negativa**, como cabía esperar: cuanto más rodado está un coche, más
barato se anuncia.

La relación **no es lineal**. El precio cae de forma abrupta en las primeras
100 000 millas y después se aplana: a partir de cierto punto, sumar millas apenas
reduce ya el precio, porque el vehículo se acerca a su valor residual.

Al recortar los valores extremos la nube se lee mucho mejor, lo que justifica el
filtro de precio incorporado a la aplicación.

## 5. Comparación por tipo de transmisión

Tercer gráfico del panel. Uso como máximo tres series para que los colores sigan
siendo distinguibles por lectores con daltonismo.

In [ ]:
# Distribución del precio según el tipo de transmisión
fig = px.histogram(
    recorte,
    x='price',
    color='transmission',
    nbins=50,
    barmode='overlay',
    opacity=0.6,
    title='Distribución del precio según la transmisión',
    labels={'price': 'Precio (USD)', 'transmission': 'Transmisión'},
    color_discrete_sequence=[AZUL, NARANJA, MORADO])

fig.update_layout(yaxis_title='Número de anuncios', bargap=0.02)
fig.show()

In [ ]:
# Resumen numérico que acompaña al gráfico
resumen = (car_data.groupby('transmission')['price']
           .agg(['count', 'median', 'mean'])
           .round(0)
           .astype(int)
           .sort_values('count', ascending=False))
print(resumen)

**Lectura de la comparación.** La transmisión **automática domina el mercado**
por volumen de anuncios. Los vehículos de transmisión manual tienden a anunciarse
a precios más bajos, aunque la diferencia se explica en buena parte por el tipo de
vehículo: las camionetas y todoterrenos de gama alta suelen ser automáticos.

## 6. Tipos de vehículo más anunciados

In [ ]:
# Número de anuncios por tipo de vehículo
tipos = car_data['type'].value_counts()

fig = px.bar(
    x=tipos.values,
    y=tipos.index,
    orientation='h',
    title='Número de anuncios por tipo de vehículo',
    labels={'x': 'Número de anuncios', 'y': 'Tipo de vehículo'},
    color_discrete_sequence=[AZUL])

fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print(tipos)

## 7. Conclusiones del análisis exploratorio

1. **El conjunto tiene 51 525 anuncios y 13 columnas.** No hay filas duplicadas y
   las columnas clave (`price`, `type`, `condition`, `transmission`) están completas.

2. **Cinco columnas tienen ausentes**, pero solo una es problemática de verdad:
   `odometer` (15,3 %). El caso de `is_4wd` no es una ausencia real sino una
   codificación implícita del valor "no".

3. **Existen valores extremos** que distorsionan la escala de los gráficos:
   precios de hasta 375 000 USD y odómetros de casi un millón de millas. La
   aplicación los gestiona mediante un filtro de precio ajustable.

4. **La relación entre kilometraje y precio es negativa y no lineal**: el valor se
   desploma en las primeras 100 000 millas y luego se estabiliza.

5. **Los tipos más anunciados son SUV, sedán y camioneta**, que juntos superan la
   mitad del mercado.

Estos hallazgos definen los tres gráficos y los filtros que incorpora la
aplicación web: histograma del kilometraje, dispersión kilometraje-precio y
comparación de precios por transmisión.